# SI — the PSF at every wavelength is the 600 nm PSF, rescaled

**Makes up:** the SI figure for the Image simulation section showing that drawing every photon's position from one
radial PSF table at $\lambda_0 = 600$ nm, rescaled by $\lambda/\lambda_0$
(`pyS3M.simulation.vectorial_sampler.RadialDipolePSF`), reproduces the PSF calculated directly at each wavelength.

**Why the rescaling holds.** For a freely rotating dipole in focus at the coverslip, the pupil field depends only on
the emission angle when the refractive indices do not depend on wavelength, and the image is its Fourier transform at
$k\,\boldsymbol{\rho}$ with $k = 2\pi\,\mathrm{NA}/\lambda$, so

$$U_\lambda(r) = \left(\frac{\lambda_0}{\lambda}\right)^2 U_{\lambda_0}\!\left(r\,\frac{\lambda_0}{\lambda}\right).$$

With fixed indices this is exact (checked below to the table's sampling). The approximation is the fixed indices, so
the reference here is the vectorial PSF (`DipolePSF`) **calculated directly at each wavelength with dispersive water
and immersion oil** (and the NA they imply), with no table or rescaling.

**Data:** none; everything is calculated. Runs in under a minute.

---
## 0 — Setup

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pyS3M import PlottingBase
from pyS3M.simulation.defocus_psf import DipolePSF
from pyS3M.simulation.vectorial_sampler import RadialDipolePSF

plotter = PlottingBase.PublicationPlotter(dark_background=False)

OUT_ROOT = Path("outputs") / "FigureSI_VectorialScaling"   # git-ignored (bare outputs/ rule)
FIG_DIR = OUT_ROOT
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The simulation's optics (Image simulation section): oil objective, dipole in water at the coverslip
NA, N_WATER, N_OIL = 1.49, 1.33, 1.515
LAMBDA0_UM = 0.6        # reference wavelength of the radial table
SHOWN_UM = [0.45, 0.55, 0.65, 0.75]
SHOWN_COLOURS = ["#1f3fff", "#1a9a1a", "#e01010", "#7a0000"]

---
## 1 — Refractive-index dispersion

Water: Sellmeier fit at 20 °C (Daimon & Masumura 2007). Oil: Cauchy, $n_e = 1.518$, Abbe number $\nu_e = 45$
(Zeiss Immersol 518F). Both curves are shifted to the simulation's 1.33 and 1.515 at 600 nm, so the two models agree
there exactly and differ elsewhere by dispersion alone. The objective keeps its aperture angle, so its NA follows
the oil: $\mathrm{NA}(\lambda) = 1.49\,n_\mathrm{oil}(\lambda)/1.515$.

In [ ]:
def n_water(lam_um):
    """Water at 20 C (Daimon & Masumura 2007, Appl. Opt. 46, 3811; Sellmeier)."""
    A = [5.684027565e-1, 1.726177391e-1, 2.086189578e-2, 1.130748688e-1]
    B = [5.101829712e-3, 1.821153936e-2, 2.620722293e-2, 1.069792721e1]
    l2 = np.asarray(lam_um, float) ** 2
    return np.sqrt(1 + sum(a * l2 / (l2 - b) for a, b in zip(A, B)))


def n_oil(lam_um):
    """Immersion oil, Cauchy n = A + B / lambda^2 with n_e (546.1 nm) = 1.518 and Abbe number
    nu_e = 45 (as Zeiss Immersol 518F)."""
    B = (0.518 / 45) / (1 / 0.48 ** 2 - 1 / 0.6438 ** 2)
    return 1.518 - B / 0.5461 ** 2 + B / np.asarray(lam_um, float) ** 2


def full_model(lam_um):
    """The PSF calculated directly at lam, with dispersive refractive indices: water and oil follow
    their dispersion curves, shifted to the simulation's 1.33 / 1.515 at 600 nm, and the objective
    keeps its aperture angle, so NA = 1.49 n_oil(lam) / 1.515."""
    n_w = N_WATER + n_water(lam_um) - n_water(LAMBDA0_UM)
    n_o = N_OIL + n_oil(lam_um) - n_oil(LAMBDA0_UM)
    return DipolePSF(NA=NA * n_o / N_OIL, n_medium=float(n_w), n_immersion=float(n_o), N_pupil=512)


fixed_n = DipolePSF(NA=NA, n_medium=N_WATER, n_immersion=N_OIL, N_pupil=512)
table = RadialDipolePSF(NA=NA, n_medium=N_WATER, n_immersion=N_OIL, N_pupil=512, ref_wavelength_um=LAMBDA0_UM)
for lam in [0.4, 0.6, 0.8]:
    m = full_model(lam)
    print(f"{lam * 1000:.0f} nm: n_water {m.n_medium:.4f}, n_oil {m.n_immersion:.4f}, NA {m.NA:.4f}")

---
## 2 — The rescaled profiles against the 600 nm table

At each wavelength, the directly calculated radial profile against the 600 nm table rescaled to that wavelength
(what the simulation draws photons from), as the largest difference relative to the peak, out to 1 µm: with the
simulation's fixed indices (the table's sampling only) and with dispersive water and oil.

In [ ]:
r_check = np.linspace(0, 1.0, 1001)   # um, at each wavelength
print("largest |rescaled profile - 600 nm table| / peak, r <= 1 um:")
for lam in np.round(np.arange(0.40, 0.8001, 0.05), 3):
    on_table = np.interp(r_check * LAMBDA0_UM / lam, table.r_um, table.density_um2) * (LAMBDA0_UM / lam) ** 2
    for name, model in [("fixed indices", fixed_n), ("dispersive", full_model(lam))]:
        prof = model.image(lam, r_check, np.zeros(1))[0]
        print(f"  {lam * 1000:.0f} nm, {name:13s}: {np.abs(on_table - prof).max() / prof.max():.1e}")

---
## 3 — Figure

a) the directly calculated (dispersive) PSF at 450, 550, 650 and 750 nm, each on its own colour scale from 0 to its
99.99th percentile (per µm² of sample plane; the PSF integrates to 1). b) their radial profiles. c) the same profiles rescaled, $r\,\lambda_0/\lambda$ and
intensity × $(\lambda/\lambda_0)^2$, on the 600 nm table (dashed): they fall on one curve.

In [ ]:
MM = 1 / 25.4
fig = plt.figure(figsize=(170 * MM, 100 * MM))
fig.set_layout_engine("none")
W, H = 170.0, 100.0


def add(x, y, w, h):   # mm from the top-left
    return fig.add_axes([x / W, 1 - (y + h) / H, w / W, h / H])


def label(ax, s, dx=-9.0, dy=-1.0):
    ax.text(dx / (ax.get_position().width * W), 1 + dy / (ax.get_position().height * H) * -1,
            f"{s})", transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")


def grid(ax):
    ax.grid(True, ls="--", color="grey", alpha=0.4, lw=0.5)


# a) the PSF calculated directly (dispersive) at four wavelengths, each scaled 0 to its 99.99th percentile
half = 0.6
xs = np.linspace(-half, half, 241)
imgs = [full_model(lam).image(lam, xs, xs) for lam in SHOWN_UM]
size, gap, top = 30.0, 9.0, 6.0
for j, (lam, img, col) in enumerate(zip(SHOWN_UM, imgs, SHOWN_COLOURS)):
    x0 = 12 + j * (size + gap)
    ax = add(x0, top, size, size)
    im = ax.imshow(img, cmap="inferno", vmin=0, vmax=np.percentile(img, 99.99), origin="lower",
                   extent=[-half, half, -half, half])
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.05, 0.95, f"{lam * 1000:.0f} nm", transform=ax.transAxes, color="white", fontsize=7, va="top")
    for s in ax.spines.values():
        s.set_color(col); s.set_linewidth(1.5)
    if j == 0:
        plotter.add_scalebar(ax, 1.0, 0.5, location="lower right", color="white", fontsize=6, label="500 nm")
        label(ax, "a")
    cb = fig.colorbar(im, cax=add(x0 + size + 0.8, top, 1.5, size))
    cb.ax.tick_params(labelsize=6)
    if j == len(SHOWN_UM) - 1:   # all four in the same unit: label the rightmost only
        cb.set_label(r"intensity ($\mu$m$^{-2}$)", fontsize=7)

# b) radial profiles; c) the same, rescaled to 600 nm, on the 600 nm table
r = np.linspace(0, 1.2, 601)
py = 52.0
ax_b = add(14, py, 66, 40)
ax_c = add(96, py, 66, 40)
for lam, col in zip(SHOWN_UM, SHOWN_COLOURS):
    prof = full_model(lam).image(lam, r, np.zeros(1))[0]
    ax_b.plot(r * 1000, prof, color=col, lw=1.2, label=f"{lam * 1000:.0f} nm")
    ax_c.plot(r * 1000 * LAMBDA0_UM / lam, prof * (lam / LAMBDA0_UM) ** 2, color=col, lw=2.2, alpha=0.6)
ax_c.plot(table.r_um * 1000, table.density_um2, color="black", ls=(0, (3, 3)), lw=0.8,
          label="600 nm table")
ax_b.set_xlim(0, 1000); ax_b.set_ylim(1e-3, 30); ax_b.set_yscale("log")
ax_c.set_xlim(0, 1000); ax_c.set_ylim(1e-3, 30); ax_c.set_yscale("log")
ax_b.set_xlabel("distance from the molecule, $r$ (nm)", fontsize=7)
ax_b.set_ylabel(r"intensity ($\mu$m$^{-2}$)", fontsize=7)
ax_c.set_xlabel(r"$r\,\lambda_0/\lambda$ (nm)", fontsize=7)
ax_c.set_ylabel(r"intensity $\times\,(\lambda/\lambda_0)^2$ ($\mu$m$^{-2}$)", fontsize=7)
ax_b.legend(fontsize=6, loc="upper right")
ax_c.legend(fontsize=6, loc="upper right")
for ax, s in [(ax_b, "b"), (ax_c, "c")]:
    ax.tick_params(labelsize=6); grid(ax); label(ax, s)

fig.savefig(FIG_DIR / "PSF_WavelengthScaling.svg", dpi=600, format="svg")
plt.show()